# 항공 데이터 학습 (코랩 실행용)

코드는 `core/`, `categories/`의 `.py` 파일을 그대로 쓰고, 이 노트북은 실행만 맡습니다. **2020년 이후 자료만 씁니다** (`core/data.py`의 `START_YEAR`).

**준비:** 이 PC에서 `upload_to_drive.ps1`로 드라이브의 `내 드라이브/4ir`에 올린 뒤, 메뉴 `런타임 > 런타임 유형 변경`에서 GPU를 고릅니다.

**세션을 새로 열 때마다 1~3번을 실행합니다.** 코랩 로컬 디스크는 세션이 끝나면 지워지지만, `outputs/`는 드라이브에 바로 저장되므로 끝난 모델은 다시 학습하지 않습니다.

## 1. 드라이브 연결과 환경 확인

In [ ]:
import importlib.util, json, os, shutil, subprocess, sys, tempfile, time, unicodedata
from pathlib import Path

DRIVE = Path("/content/drive/MyDrive/4ir")
LOCAL = Path("/content/4ir")

from google.colab import drive
drive.mount("/content/drive")
if not (DRIVE / "run_models.py").exists():
    raise SystemExit(f"{DRIVE} 에 프로젝트가 없습니다. 먼저 upload_to_drive.ps1 로 올리세요.")

ram_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1024**3
print(f"CPU {os.cpu_count()}개, 메모리 {ram_gb:.0f}GB, 로컬 디스크 여유 {shutil.disk_usage('/content').free / 1024**3:.0f}GB")
gpu = shutil.which("nvidia-smi") and subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "없음. LSTM 을 돌리려면 런타임 유형을 GPU 로 바꾸세요.")

## 2. 코드와 데이터를 로컬 디스크로 복사

학습 데이터는 디스크 배열을 무작위로 읽기 때문에 드라이브에서 직접 읽으면 매우 느립니다. 같은 세션에서 다시 실행하면 바뀐 파일만 복사합니다.

In [ ]:
def sync(src, dst, skip=()):
    """src 를 dst 로 복사한다. 크기가 같고 dst 가 오래되지 않은 파일은 건너뛴다. 끊겨도 잘린 파일이 남지 않게 임시 이름으로 복사한다.

    코드의 한글 파일 이름은 NFC 인데 드라이브 연결이 NFD 로 돌려줄 수 있어, 복사본 이름을 NFC 로 맞춘다.
    """
    # meta.json 은 캐시 완성 표시이므로 마지막에 복사한다. 먼저 복사하면 끊겼을 때 미완성 캐시를 완성본으로 읽는다
    files = sorted((p for p in src.rglob("*") if p.is_file()), key=lambda p: (p.name == "meta.json", str(p)))
    copied = 0
    for f in files:
        rel = f.relative_to(src)
        if rel.parts[0] in skip or "__pycache__" in rel.parts or f.name.endswith(".copying"):
            continue
        target = dst / unicodedata.normalize("NFC", str(rel))
        s = f.stat()
        if target.exists() and target.stat().st_size == s.st_size and target.stat().st_mtime >= s.st_mtime - 2:
            continue
        target.parent.mkdir(parents=True, exist_ok=True)
        tmp = target.with_name(target.name + ".copying")
        shutil.copy2(f, tmp)
        tmp.replace(target)
        copied += s.st_size
    return copied


with tempfile.TemporaryDirectory() as t:  # 실제 복사 전에 규칙을 점검한다
    a, b = Path(t, "a"), Path(t, "b")
    (a / "skip").mkdir(parents=True)
    (a / "meta.json").write_text("m")
    (a / "skip" / "x").write_text("x")
    assert sync(a, b, skip=("skip",)) == 1 and (b / "meta.json").exists() and not (b / "skip").exists()
    assert sync(a, b, skip=("skip",)) == 0, "바뀌지 않은 파일은 다시 복사하지 않음"
    (a / "meta.json").write_text("2")
    os.utime(a / "meta.json", (time.time() + 60, time.time() + 60))
    assert sync(a, b, skip=("skip",)) == 1 and (b / "meta.json").read_text() == "2", "크기가 같아도 더 새 파일이면 복사"
    nfc, nfd = "한글.csv", unicodedata.normalize("NFD", "한글.csv")
    (a / nfd).write_text("k", encoding="utf-8")
    sync(a, b, skip=("skip",))
    assert nfc in {p.name for p in b.iterdir()}, "NFD 이름은 NFC 로 복사"

start = time.time()
LOCAL.mkdir(parents=True, exist_ok=True)
print(f"복사 {sync(DRIVE, LOCAL, skip=('outputs', 'outputs_2026', '_archive_before_2020')) / 1024**3:.1f}GB, {(time.time() - start) / 60:.0f}분")

# 결과는 드라이브에 바로 저장해 세션이 끊겨도 남긴다. Store 와 run_models 의 이어 하기가 이 폴더를 본다
for name in ("outputs", "outputs_2026"):
    (DRIVE / name).mkdir(exist_ok=True)
    outputs = LOCAL / name
    if not outputs.is_symlink():
        if outputs.exists():
            raise SystemExit(f"{outputs} 가 이미 일반 폴더입니다. 결과를 잃지 않도록 내용을 확인한 뒤 옮기세요.")
        outputs.symlink_to(DRIVE / name, target_is_directory=True)

os.chdir(LOCAL)  # prepare_data.py 와 us_flights 가 data/ 를 현재 폴더 기준으로 읽는다
print("작업 폴더:", os.getcwd(), "/ 결과 폴더:", (LOCAL / "outputs").resolve())

## 3. 패키지 확인

`requirements.txt`는 Windows에서 고정한 버전이라 설치하지 않습니다. 코랩의 TensorFlow GPU 환경을 유지하고 없는 패키지만 설치합니다.

In [ ]:
NEEDED = {"lightgbm": "lightgbm", "xgboost": "xgboost", "pyarrow": "pyarrow", "openpyxl": "openpyxl", "sklearn": "scikit-learn", "tensorflow": "tensorflow"}
missing = [pip for name, pip in NEEDED.items() if importlib.util.find_spec(name) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

from importlib.metadata import version
for pip in NEEDED.values():
    print(pip, version(pip))
print("pandas", version("pandas"), "/ numpy", version("numpy"), "(이 PC는 pandas 3.1.0, numpy 2.5.4)")

## 4. 동작과 속도 확인

`trial_model.py`는 결과를 임시 폴더에 쓰므로 `outputs/`의 기존 결과는 바뀌지 않습니다.

가장 작은 과제부터 돌립니다. 학습 데이터를 섞을 때 시드를 고정하지 않아서 같은 PC에서도 MAE가 10%쯤 달라집니다(5,622와 5,105). 오류가 나거나 자릿수가 달라질 때만 라이브러리 버전 차이를 의심하세요.

In [ ]:
!python trial_model.py korea_demand type_passengers

## 5. 2026년 자료로 구동 확인

4개 모델(LightGBM, XGBoost, RandomForest, LSTM)을 **2026년 행으로만** 학습합니다. 자료가 있는 달 중 마지막 25%를 평가, 그 앞 한 달을 검증, 나머지를 학습에 씁니다. 2026년 자료가 없는 과제는 건너뜁니다.

결과는 드라이브의 `outputs_2026/`에 따로 저장되어 전체 학습과 섞이지 않습니다. 끊기면 1~3번을 다시 실행한 뒤 같은 셀을 실행하세요. 끝난 모델은 건너뜁니다.

**구동 확인용입니다.** 겨울~봄으로 학습하고 여름을 평가하므로, 성능 수치를 최종 성능으로 읽으면 안 됩니다.

In [ ]:
!python run_models.py --year=2026 europe korea_demand korea_ops safety us_flights us_weather 2>&1 | tee -a "{DRIVE}/run_models_2026.log"

In [ ]:
import pandas as pd

rows = [json.loads(f.read_text(encoding="utf-8")) for f in Path("outputs_2026").glob("*/model_results/*.json")]
result = pd.DataFrame(rows)
print(f"완료 {len(result)}개 모델")
if len(result):
    display(pd.crosstab(result["task"], result["model"]))
    shown = ["task", "model", "n_train", "n_test", "train_period", "test_period", "seconds", "roc_auc", "pr_auc", "mae", "baseline_mae", "r2", "accuracy", "macro_f1"]
    display(result[[c for c in shown if c in result]].sort_values(["task", "model"]).round(3))

## 6. 전체 학습 (2020년 이후)

끝난 모델은 자동으로 건너뜁니다. 세션이 끊기면 1~3번을 다시 실행한 뒤 같은 셀을 실행하세요. 진행 중이던 과제만 처음부터 다시 합니다. 로그는 드라이브의 `run_models_colab.log`에 남습니다.

### 6-1. BTS 제외 카테고리

4개 모델을 모두 학습합니다. 2020년 기준으로 바꾸면서 옛 결과를 옮겨 두었으므로 다시 학습이 필요합니다. 트리 모델은 GPU를 쓰지 않습니다.

In [ ]:
!python run_models.py europe korea_demand korea_ops safety us_weather world_traffic 2>&1 | tee -a "{DRIVE}/run_models_colab.log"

### 6-2. BTS 디스크 캐시 확인 (없을 때만 생성)

드라이브에 `data/cache/bts_2020`을 올려 두었다면 2번 셀에서 이미 복사되어 바로 끝납니다. 없으면 약 4,300만 행으로 약 8GB를 만들고 드라이브에도 보관합니다.

In [ ]:
meta = Path("data/cache/bts_2020/meta.json")
if not meta.exists():
    !python -c "from categories.us_flights.models import build; build()"
if not meta.exists():
    raise SystemExit("캐시를 만들지 못했습니다. 위 출력을 확인하세요.")
print(f"드라이브에 저장 {sync(meta.parent, DRIVE / 'data/cache/bts_2020') / 1024**3:.1f}GB")

### 6-3. BTS LSTM 11개

이 PC 속도로 단순 계산하면 큰 과제 하나에 약 30분입니다. 5번의 실행 시간을 보고 판단하세요.

In [ ]:
!python run_models.py --models=lstm us_flights 2>&1 | tee -a "{DRIVE}/run_models_colab.log"

### 6-4. BTS 트리 모델

입력 데이터만 약 2GB를 메모리에 올립니다. 세션이 강제 종료되면 메모리가 더 큰 런타임으로 바꾸세요.

In [ ]:
!python run_models.py --models=lightgbm,xgboost,random_forest us_flights 2>&1 | tee -a "{DRIVE}/run_models_colab.log"

## 7. 진행 현황과 마무리

결과 앱은 `outputs/`만 있으면 동작합니다. 끝나면 드라이브의 `4ir/outputs` 폴더를 이 PC의 프로젝트로 복사해 Streamlit으로 보세요.

In [ ]:
import pandas as pd

rows = [{"카테고리": f.parent.parent.name, "모델": json.loads(f.read_text(encoding="utf-8"))["model"]} for f in Path("outputs").glob("*/model_results/*.json")]
display(pd.crosstab(pd.DataFrame(rows)["카테고리"], pd.DataFrame(rows)["모델"]))

In [ ]:
drive.flush_and_unmount()  # 세션을 닫기 전에 실행하면 드라이브에 쓰는 중인 내용을 끝까지 저장한다